# <div align="center"><b> PROCESAMIENTO EN BATCH IMÁGENES GEORREFERENCIADAS </b></div>

<div align="right">

<!-- [![Binder](http://mybinder.org/badge.svg)](https://mybinder.org/) -->
[![nbviewer](https://img.shields.io/badge/render-nbviewer-orange?logo=Jupyter)](https://nbviewer.org)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com)

</div>

* * *

<style>
/* Limitar la altura de las celdas de salida en html */
.jp-OutputArea.jp-Cell-outputArea {
    max-height: 500px;
}
</style>

<!-- Descargar archivos adicionales:
!gdown https://drive.google.com/drive/folders/1UBZ8PEbtmiWMGkULu7GAt3VhUpeTy9l7?usp=sharing --folder -->

In [1]:
# Chequear versión de CUDA
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Wed_Oct_30_01:18:48_Pacific_Daylight_Time_2024
Cuda compilation tools, release 12.6, V12.6.85
Build cuda_12.6.r12.6/compiler.35059454_0


In [2]:
# Chequear más datos sobre la GPU
!nvidia-smi

Sat Mar 21 16:55:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.57                 Driver Version: 581.57         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
|  0%   37C    P8              5W /  320W |     503MiB /  16376MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

✋ <em><font color='DodgerBlue'>Importaciones:</font></em> ✋

In [3]:
# Recarga automática de módulos en Jupyter Notebook
%reload_ext autoreload
%autoreload 2

In [4]:
# Dependencias del sistema
from pathlib import Path
import os, json, re
from typing import Optional

# Dependencias locales
from modulo_ia.config import settings as CONFIG
from modulo_ia.core.modeling.predict import DetectionModelPredictor
from modulo_ia.core.modeling.predict import PredictionResult
from modulo_utilidades.core.labeling import procesador_geojson_kml_core as procesador_geojson_kml

# Dependencias propias

# Dependencias de terceros
from loguru import logger as LOGGER
from ultralytics import YOLO
from tqdm.auto import tqdm
import cv2
import pandas as pd
import geopandas as gpd

os.environ["FIFTYONE_PLUGINS_DIR"] = (
    "E:\\Documentos\\Git Repositories\\uba-ceia-proy-final\\ceia-proyecto-final\\modulo-IA\\.venv\\Lib\\site-packages\\plugins\\operators"
)
import fiftyone as fo

fo.config.plugins_dir = "E:\\Documentos\\Git Repositories\\uba-ceia-proy-final\\ceia-proyecto-final\\modulo-IA\\.venv\\Lib\\site-packages\\plugins\\operators"

2026-03-21 16:55:08.916 | DEBUG    | modulo_ia.config:<module>:13 - Loading modulo-ia config...
2026-03-21 16:55:08.916 | WARNING  | modulo_ia.config:<module>:24 - Using .env file at E:\Documentos\Git Repositories\uba-ceia-proy-final\ceia-proyecto-final\modulo-IA\.env for configuration.
E:\Documentos\Git Repositories\uba-ceia-proy-final\ceia-proyecto-final\modulo-IA\modulo_ia\config.py:75: UserWarning: Field name "schema" in "FiftyoneConfig" shadows an attribute in parent "BaseModel"
  class FiftyoneConfig(BaseModel):
E:\Documentos\Git Repositories\uba-ceia-proy-final\ceia-proyecto-final\modulo-IA\modulo_ia\config.py:86: UserWarning: Field name "schema" in "MLFlowConfig" shadows an attribute in parent "BaseModel"
  class MLFlowConfig(BaseModel):
2026-03-21 16:55:08.924 | DEBUG    | modulo_ia.config:<module>:120 - Settings (modulo-ia) loaded: {
  "environment": "dev",
  "seed": 42,
  "folders": {
    "project_dir": "E:\\Documentos\\Git Repositories\\uba-ceia-proy-final\\ceia-proyecto-fi

🔧 <em><font color='tomato'>Configuraciones:</font></em> 🔧


In [5]:
MODEL_FOLDER = CONFIG.folders.models_folder
MODEL_NAME = "rpw_detection_yolo11x_640_freeze_learning_5008788e1d99471e97b877bca45f169f.pt"
MODEL_PATH = MODEL_FOLDER / MODEL_NAME

DOWNLOAD_FOLDER = Path("downloads")
PREDICTIONS_FOLDER = Path("predicciones")

# PREDICCINES
NMS_IOU_THRESHOLD = 0.1
NMM_IOU_THRESHOLD = 0.1
CONFIDENCE = 0.5
MIN_RATIO = 0.7
CONTAINERMENT_THRESHOLD = 0.8
OVERLAP_RATIO_WH = (0.1, 0.1)
IMG_SIZE = 640  # Tamaño de la imagen
OVERLAP_RATIO_WH = (0.1, 0.1)

<div align="center">✨Datos del proyecto:✨</div>

<p></p>

<div align="center">

| **Subtitulo**   | Predicciones masivas con georreferenciación                                                                                                        |
| --------------- | -------------------------------------------------------------------------------------------------------------------------------------- |
| **Descrpción**  | <small>Notebook de detección de palmeras afectadas por el picudo rojo de forma masiva</small>                                                                    |

</div>

## TABLA DE CONTENIDO

1. [Modelo del proceso](#modelo-del-proceso)

## MODELO DEL PROCESO

1. Inicialmente, se cargan las imágenes desde la carpeta de descargas ("downloads"). Esta carpeta contiene las imágenes y los archivos de georreferenciación asociados a cada imagen, del tipo .jgw.
2. Se carga el modelo de detección previamente entrenado, utilizando la ruta especificada en la variable `MODEL_PATH`.
3. Se itera sobre cada imagen en la carpeta de descargas, realizando las siguientes operaciones para cada imagen:
   - Se carga la imagen utilizando OpenCV.
   - Se realiza la predicción utilizando el modelo de detección cargado.
   - Se extraen las coordenadas de las predicciones y se convierten a coordenadas geográficas utilizando la información de georreferenciación proporcionada por el archivo .jgw correspondiente a cada imagen.
   - Se guardan las predicciones con sus respectivas coordenadas geográficas en formato KML y GeoJSON.

## CARGAR IMÁGENES Y GEOREFERENCIACIÓN

In [6]:
download_folder = DOWNLOAD_FOLDER.resolve()
imgs_path = download_folder.glob("*.jpg")
georref_path = download_folder.glob("*.jgw")

data = [
    {"id": idx, "img_name": img.name, "img_path": str(img), "georref_path": str(georef)}
    for idx, (img, georef) in enumerate(zip(imgs_path, georref_path))
]

LOGGER.info(f"Se encontraron {len(data)} imágenes y archivos de georreferenciación en la carpeta de descargas.")
LOGGER.info(f"Ejemplo de datos cargados: {data[0]}")

2026-03-21 16:55:13.183 | INFO     | __main__:<module>:10 - Se encontraron 47 imágenes y archivos de georreferenciación en la carpeta de descargas.
2026-03-21 16:55:13.183 | INFO     | __main__:<module>:11 - Ejemplo de datos cargados: {'id': 0, 'img_name': 'RGB_Minas_2025_G26C6P4.jpg', 'img_path': 'E:\\Documentos\\Git Repositories\\uba-ceia-proy-final\\ceia-proyecto-final\\modulo-IA\\notebooks\\anotaciones_masivas\\downloads\\RGB_Minas_2025_G26C6P4.jpg', 'georref_path': 'E:\\Documentos\\Git Repositories\\uba-ceia-proy-final\\ceia-proyecto-final\\modulo-IA\\notebooks\\anotaciones_masivas\\downloads\\RGB_Minas_2025_G26C6P4.jgw'}


## CARGAR MODELO DE DETECCIÓN

In [7]:
model = YOLO(MODEL_PATH)

## REALIZAR PREDICCIONES

In [8]:
def parse_jgw_data(jgw_path: Path) -> dict[str, float]:
    with open(jgw_path, "r") as f:
        lines = f.readlines()
        if len(lines) != 6:
            raise ValueError(f"El archivo {jgw_path} no tiene el formato correcto (debe contener 6 líneas).")
        try:
            pixel_size_x = float(lines[0].strip())
            rotation_x = float(lines[1].strip())
            rotation_y = float(lines[2].strip())
            pixel_size_y = float(lines[3].strip())
            upper_left_x = float(lines[4].strip())
            upper_left_y = float(lines[5].strip())
        except ValueError as e:
            raise ValueError(f"Error al convertir los valores del archivo {jgw_path} a números: {e}")
    return {
        "x_pixel_size": pixel_size_x,
        "y_rotation": rotation_y,
        "x_rotation": rotation_x,
        "y_pixel_size": pixel_size_y,
        "x_origin": upper_left_x,
        "y_origin": upper_left_y,
    }

In [9]:
def predict_one_image(
    model_predictor: DetectionModelPredictor, img_path: Path, jgw_path: Path, output_folder: Path
) -> None:
    LOGGER.info(f"➡️ Procesando imagen: {img_path}")
    if not img_path.exists():
        LOGGER.error(f"El archivo de imagen no existe: {img_path}")
        raise FileNotFoundError(f"El archivo de imagen no existe: {img_path}")
    if not jgw_path.exists():
        LOGGER.error(f"El archivo de georreferenciación no existe: {jgw_path}")
        raise FileNotFoundError(f"El archivo de georreferenciación no existe: {jgw_path}")
    output_folder.mkdir(parents=True, exist_ok=True)

    img_name = img_path.stem
    prediction: PredictionResult = (
        model_predictor.predict(img_path)
        .filter_by_square_ratio(min_ratio=MIN_RATIO)
        .filter_by_nms(NMS_IOU_THRESHOLD, class_agnostic=True)
        .filter_by_containment(CONTAINERMENT_THRESHOLD, class_agnostic=True, check_confidence=False)
        .filter_by_confidence(CONFIDENCE)
    )
    LOGGER.debug(f"---- ✅ Predicciones predicciones realizadas con éxito.")

    annotated_image = prediction.get_annotated_image(cv2.imread(str(img_path)))
    annotated_image_path = output_folder / f"{img_name}_predicted.jpg"
    cv2.imwrite(str(annotated_image_path), annotated_image)
    LOGGER.debug(f"---- ✅ Imagen anotada guardada en: {annotated_image_path}")

    coco_annotations = prediction.as_coco_annotations(img_name)
    LOGGER.debug(f"---- ✅ Anotaciones convertidas a formato COCO.")

    jgw_data = parse_jgw_data(jgw_path)
    LOGGER.debug(f"---- ✅ Archivo JGW parseado correctamente.")

    gdf_path = output_folder / f"{img_name}_predicted.geojson"
    gdf = procesador_geojson_kml.create_geojson_from_annotations(img_name, coco_annotations, jgw_data, gdf_path)
    LOGGER.debug(f"---- ✅ GeoDataFrame procesado.")

    kml_path = output_folder / f"{img_name}_predicted.kml"
    procesador_geojson_kml.generate_kml_from_geojson(gdf, output_file_path=kml_path)
    LOGGER.debug(f"---- ✅ Archivo KML procesado.")
    LOGGER.info(f"---- ✅ Procesamiento completo para la imagen: {img_path}")

In [10]:
model_predictor = DetectionModelPredictor(
    model=model, target_img_size_wh=(IMG_SIZE, IMG_SIZE), overlap_ratio_wh=OVERLAP_RATIO_WH
)

In [11]:
LOGGER.level("INFO")
for item in tqdm(data, desc="Procesando imágenes"):
    predict_one_image(
        model_predictor=model_predictor,
        img_path=Path(item["img_path"]),
        jgw_path=Path(item["georref_path"]),
        output_folder=PREDICTIONS_FOLDER,
    )

Procesando imágenes:   0%|          | 0/47 [00:00<?, ?it/s]2026-03-21 16:55:13.658 | INFO     | __main__:predict_one_image:4 - ➡️ Procesando imagen: E:\Documentos\Git Repositories\uba-ceia-proy-final\ceia-proyecto-final\modulo-IA\notebooks\anotaciones_masivas\downloads\RGB_Minas_2025_G26C6P4.jpg
2026-03-21 16:55:30.334 | DEBUG    | __main__:predict_one_image:21 - ---- ✅ Predicciones predicciones realizadas con éxito.
2026-03-21 16:55:32.460 | DEBUG    | __main__:predict_one_image:26 - ---- ✅ Imagen anotada guardada en: predicciones\RGB_Minas_2025_G26C6P4_predicted.jpg
2026-03-21 16:55:32.460 | DEBUG    | modulo_utilidades.core.labeling.procesador_anotaciones_coco_dataset_core:create_coco_annotations_from_detections:67 - No se proporcionó un mapa de categorías. Se utilizará el mapa de categorías predeterminado del dataset COCO. Categorías: {categories}
2026-03-21 16:55:32.460 | DEBUG    | __main__:predict_one_image:29 - ---- ✅ Anotaciones convertidas a formato COCO.
2026-03-21 16:55:32.

## POST-PROCESAMIENTO

### CREAR GEOJSON TOTAL

In [12]:
def merge_geojson_files(
    geojson_paths: list[Path | str],
    output_file_path: Optional[Path] = None,
    target_epsg: str = "EPSG:4326",
    drop_duplicates: bool = False,
    subset_duplicates: Optional[list[str]] = None,
) -> gpd.GeoDataFrame:
    """
    Une múltiples archivos GeoJSON en un único GeoDataFrame.

    Args:
        geojson_paths: Lista de rutas a archivos .geojson.
        output_file_path: Si se indica, guarda el merge en disco.
        target_epsg: CRS objetivo común para todos los archivos.
        drop_duplicates: Si True, elimina duplicados.
        subset_duplicates: Columnas para evaluar duplicados. Si None, usa todas.

    Returns:
        GeoDataFrame unificado. Si no hay datos válidos, retorna GeoDataFrame vacío.
    """
    if not geojson_paths:
        LOGGER.warning("No se recibieron rutas de GeoJSON para unir.")
        return gpd.GeoDataFrame()

    gdfs: list[gpd.GeoDataFrame] = []

    for path in geojson_paths:
        p = Path(path)

        if not p.exists():
            LOGGER.warning(f"Archivo no encontrado, se omite: {p}")
            continue

        try:
            gdf = gpd.read_file(p)
        except Exception as e:
            LOGGER.warning(f"No se pudo leer {p}. Error: {e}")
            continue

        if gdf.empty:
            LOGGER.info(f"GeoJSON vacío, se omite: {p}")
            continue

        # Si no tiene CRS, asumimos target_epsg para poder unificar.
        if gdf.crs is None:
            gdf = gdf.set_crs(target_epsg, allow_override=True)
        elif str(gdf.crs).upper() != target_epsg.upper():
            gdf = gdf.to_crs(target_epsg)

        gdfs.append(gdf)

    if not gdfs:
        LOGGER.warning("No hubo GeoJSONs válidos para unir.")
        return gpd.GeoDataFrame()

    merged_df = pd.concat(gdfs, ignore_index=True)
    merged_gdf = gpd.GeoDataFrame(merged_df, geometry="geometry", crs=target_epsg)

    if drop_duplicates:
        merged_gdf = merged_gdf.drop_duplicates(subset=subset_duplicates).reset_index(drop=True)

    # Reasignar annotation_id como secuencial global único
    if "annotation_id" in merged_gdf.columns:
        merged_gdf["annotation_id"] = range(len(merged_gdf))

    if output_file_path:
        output_file_path.parent.mkdir(parents=True, exist_ok=True)
        merged_gdf.to_file(output_file_path, driver="GeoJSON")
        LOGGER.info(f"GeoJSON unificado guardado en: {output_file_path}")

    return merged_gdf

In [13]:
def replace_name_column_values(
    gdf: gpd.GeoDataFrame,
    new_name: str,
    output_file_path: Optional[Path] = None,
    column_name: str = "name",
    inplace: bool = False,
) -> gpd.GeoDataFrame:
    """
    Reemplaza todos los valores de una columna por un único valor.
    Opcionalmente guarda el resultado en disco.

    Args:
        gdf: GeoDataFrame de entrada.
        new_name: Nuevo valor para todas las filas de la columna.
        output_file_path: Ruta de salida (ej: Path("salida.geojson")).
        column_name: Nombre de la columna a actualizar. Por defecto "name".
        inplace: Si True, modifica el gdf original.

    Returns:
        GeoDataFrame actualizado.
    """
    if gdf is None:
        raise ValueError("El GeoDataFrame recibido es None.")

    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError("El argumento gdf debe ser un GeoDataFrame.")

    if column_name not in gdf.columns:
        raise ValueError(f"La columna '{column_name}' no existe en el GeoDataFrame.")

    target_gdf = gdf if inplace else gdf.copy()

    if target_gdf.empty:
        LOGGER.warning("El GeoDataFrame está vacío. No hay filas para actualizar.")
    else:
        target_gdf[column_name] = new_name
        LOGGER.info(f"Se actualizó la columna '{column_name}' con el valor '{new_name}'.")

    if output_file_path is not None:
        output_file_path.parent.mkdir(parents=True, exist_ok=True)
        target_gdf.to_file(output_file_path, driver="GeoJSON")
        LOGGER.info(f"GeoJSON guardado en: {output_file_path}")

    return target_gdf

In [14]:
geojson_paths = list(PREDICTIONS_FOLDER.glob("*.geojson"))
full_geojson_path = PREDICTIONS_FOLDER / "palmeras_minas.geojson"
temp_gpd = merge_geojson_files(
    geojson_paths=geojson_paths,
    drop_duplicates=True,
)
full_gpd = replace_name_column_values(gdf=temp_gpd, new_name="palmera", output_file_path=full_geojson_path, inplace=True)

2026-03-21 17:10:43.069 | INFO     | __main__:replace_name_column_values:37 - Se actualizó la columna 'name' con el valor 'palmera'.
2026-03-21 17:10:43.095 | INFO     | __main__:replace_name_column_values:42 - GeoJSON guardado en: predicciones\palmeras_minas.geojson


### CREAR KML TOTAL

In [15]:
full_kml_path = PREDICTIONS_FOLDER / "palmeras_minas.kml"
procesador_geojson_kml.generate_kml_from_geojson(full_gpd, output_file_path=full_kml_path)

2026-03-21 17:10:43.300 | INFO     | modulo_utilidades.core.labeling.procesador_geojson_kml_core:generate_kml_from_geojson:238 - Archivo KML guardado en predicciones\palmeras_minas.kml


fastkml.kml.KML(ns='{http://www.opengis.net/kml/2.2}', name_spaces={'kml': '{http://www.opengis.net/kml/2.2}', 'atom': '{http://www.w3.org/2005/Atom}', 'gx': '{http://www.google.com/kml/ext/2.2}'}, features=[fastkml.containers.Document(ns='{http://www.opengis.net/kml/2.2}', name_spaces={'kml': '{http://www.opengis.net/kml/2.2}', 'atom': '{http://www.w3.org/2005/Atom}', 'gx': '{http://www.google.com/kml/ext/2.2}'}, id='docid', target_id='', name=None, visibility=None, isopen=None, atom_link=None, atom_author=None, address=None, phone_number=None, snippet=None, description='PalmTrees', view=None, times=None, style_url=None, styles=[], region=None, extended_data=None, features=[fastkml.containers.Folder(ns='{http://www.opengis.net/kml/2.2}', name_spaces={'kml': '{http://www.opengis.net/kml/2.2}', 'atom': '{http://www.w3.org/2005/Atom}', 'gx': '{http://www.google.com/kml/ext/2.2}'}, id='palmeras_folder', target_id='', name='Palmeras', visibility=None, isopen=None, atom_link=None, atom_auth